# optimizer-class-dispatch — worked example 1: Dispatch table with four optimizer classes

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `optimizer-class-dispatch`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

A dispatch dictionary maps string names to optimizer classes, allowing a training script to select its optimizer from a config file or command-line argument without any `if/elif` chains. Looking up a class by name and calling it with `(params, lr=lr)` is the standard pattern. Adding a new optimizer requires only one new entry in the dictionary.

## Worked solution

**Step 1 — Define the dispatch dictionary.**
We create a module-level dict mapping lowercase string keys to `torch.optim` classes. The values are the classes themselves (not instances), so we can call them later like any constructor.

**Step 2 — Look up the class.**
`cls = OPTIM_MAP[name]` retrieves the class. If `name` is not in the dict, Python raises a `KeyError` automatically, which is the desired behavior for unknown names.

**Step 3 — Instantiate with params and lr.**
`cls(params, lr=lr)` calls the optimizer's `__init__`. This works identically for SGD, Adam, AdamW, and RMSprop because they all accept `(params, lr=lr)` as their minimum signature.

**Step 4 — Verify the type.**
We check that the returned object is an instance of `torch.optim.Optimizer`, and that it is specifically the class we requested.

In [ ]:
import torch as t
import torch.nn as nn

OPTIM_MAP = {
    'sgd':     t.optim.SGD,
    'adam':    t.optim.Adam,
    'adamw':   t.optim.AdamW,
    'rmsprop': t.optim.RMSprop,
}

def build_optimizer(name: str, params, lr: float) -> t.optim.Optimizer:
    cls = OPTIM_MAP[name]
    return cls(params, lr=lr)

# --- exercise it ---
t.manual_seed(0)
model = nn.Linear(8, 4)

for name, expected_cls in OPTIM_MAP.items():
    opt = build_optimizer(name, model.parameters(), lr=1e-3)
    print(f'{name:8s} -> {type(opt).__name__} (Optimizer: {isinstance(opt, t.optim.Optimizer)})')
    assert isinstance(opt, expected_cls), f'Expected {expected_cls.__name__}, got {type(opt).__name__}'

try:
    build_optimizer('nesterov', model.parameters(), lr=1e-3)
    assert False, 'should have raised KeyError'
except KeyError:
    print('Unknown name correctly raises KeyError')